In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

try:
    plt.style.use('seaborn-v0_8-darkgrid')
except OSError:
    plt.style.use('seaborn-darkgrid')
    

In [ ]:
# ── GENERACIÓN DE LA SEÑAL TÉRMICA (código dado) ────────────────
np.random.seed(42)
x = np.linspace(-5, 5, 100)
y = np.linspace(-5, 5, 80)
X, Y = np.meshgrid(x, y)

senal_limpia = (np.exp(-(X**2 + Y**2) / 5) * 100
                + 0.4 * np.sin(X) * np.cos(Y) * 50
                + 0.2 * np.exp(-((X - 2)**2 + (Y - 1)**2) / 3) * 30)

ruido = np.random.normal(0, 15, senal_limpia.shape)
A_observada = senal_limpia + ruido

# ── CÓDIGO DEL ESTUDIANTE ────────────────────────────────────────
plt.figure(figsize=(7, 5))
im = plt.imshow(A_observada, cmap='hot', aspect='auto')
plt.colorbar(im, label='Temperatura (u.a.)')
plt.title('Matriz observada A — Señal térmica con ruido gaussiano')
plt.xlabel('Índice columna (sensor x)')
plt.ylabel('Índice fila (sensor y)')
plt.tight_layout()
plt.show()

In [ ]:
# ── CÓDIGO DEL ESTUDIANTE ────────────────────────────────────────
# 1. SVD reducida (economic): evita calcular columnas/filas redundantes
U, S, VT = np.linalg.svd(A_observada, full_matrices=False)

print(f"Dimensiones — U: {U.shape}, S: {S.shape}, VT: {VT.shape}")
# Esperado: U=(80,80), S=(80,), VT=(80,100)  ← full_matrices=False da min(m,n)

# ── Scree Plot (código dado, se ejecuta después de definir S) ────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(S, marker='o', ms=4, linestyle='-', color='steelblue')
axes[0].set_yscale('log')
axes[0].set_title('Espectro de Valores Singulares (escala log)')
axes[0].set_ylabel('Valor Singular σᵢ')
axes[0].set_xlabel('Índice i')

energia_acum = np.cumsum(S**2) / np.sum(S**2)
axes[1].plot(energia_acum * 100, color='darkorange')
axes[1].axhline(95, linestyle='--', color='gray', label='95% energía')
axes[1].set_title('Energía Acumulada (%)')
axes[1].set_ylabel('% Energía capturada')
axes[1].set_xlabel('Número de componentes k')
axes[1].legend()

plt.tight_layout()
plt.show()

# ── Respuesta a la pregunta guiada ──────────────────────────────
k_95 = np.argmax(energia_acum >= 0.95) + 1  # +1 porque índice base-0
print(f"Componentes necesarios para capturar el 95% de energía: {k_95}")
print(f"Energía capturada con k=3: {energia_acum[2]*100:.2f}%")
print(f"Energía capturada con k=5: {energia_acum[4]*100:.2f}%")

In [ ]:
# Elegir k basado en el análisis del Scree Plot
k = 5

# ── CÓDIGO DEL ESTUDIANTE ────────────────────────────────────────
# 1. Truncar los tres factores a los primeros k componentes
U_k  = U[:, :k]          # (80, k)
S_k  = S[:k]             # (k,)
VT_k = VT[:k, :]         # (k, 100)

# 2. Reconstruir la aproximación de rango k
A_k = U_k @ np.diag(S_k) @ VT_k   # (80, 100)

# ── Visualización comparativa (código dado) ──────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, data, title in zip(
        axes,
        [senal_limpia, A_observada, A_k],
        ['Señal limpia (ground truth)',
         f'A observada (con ruido)',
         f'A_k filtrada (k={k})']):
    im = ax.imshow(data, cmap='hot', aspect='auto')
    ax.set_title(title)
    plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

# ── Métricas cuantitativas (código dado) ─────────────────────────
error_rel  = np.linalg.norm(A_observada - A_k, 'fro') / np.linalg.norm(A_observada, 'fro')
energia_cap = np.sum(S[:k]**2) / np.sum(S**2) * 100
snr_mejora  = 10 * np.log10(np.var(A_observada) / np.var(A_observada - A_k))

print(f"k utilizado          : {k}")
print(f"Energía capturada    : {energia_cap:.2f} %")
print(f"Error relativo (||·||_F): {error_rel:.4f}")
print(f"Mejora SNR estimada  : {snr_mejora:.2f} dB")

In [ ]:
# ── CÓDIGO DEL ESTUDIANTE ────────────────────────────────────────
valores_k = [1, 2, 3, 5, 10, 20, 50]
resultados = []

for k_exp in valores_k:
    A_k_exp = U[:, :k_exp] @ np.diag(S[:k_exp]) @ VT[:k_exp, :]
    
    err_rel   = np.linalg.norm(A_observada - A_k_exp, 'fro') / np.linalg.norm(A_observada, 'fro')
    energ_cap = np.sum(S[:k_exp]**2) / np.sum(S**2) * 100
    snr_est   = 10 * np.log10(np.var(A_observada) / np.var(A_observada - A_k_exp))
    
    resultados.append({
        'k': k_exp,
        'Energía capturada (%)': round(energ_cap, 2),
        'Error relativo (Frobenius)': round(err_rel, 4),
        'Mejora SNR (dB)': round(snr_est, 2)
    })

df_resultados = pd.DataFrame(resultados)
print(df_resultados.to_string(index=False))


In [ ]:
# =============================================================================
# SECCIÓN 5 — PREGUNTAS DE ANÁLISIS
#
# Pregunta 1 — Interpretación geométrica de U, Σ y Vᵀ
#
# La SVD descompone la transformación A·x en tres pasos geométricos sucesivos:
#
# - Vᵀ (rotación en el espacio de entrada): reescribe el vector x en la base
#   de los vectores singulares derechos {vᵢ}. Cada vᵢ identifica una dirección
#   de activación en el espacio original de los datos.
#
# - Σ (estiramiento): escala cada componente por el valor singular σᵢ
#   correspondiente. Los σᵢ cuantifican cuánta energía lleva cada modo;
#   σᵢ² es proporcional a la varianza capturada por ese modo.
#
# - U (rotación en el espacio de salida): mapea los modos estirados al espacio
#   de salida usando la base de vectores singulares izquierdos {uᵢ}, que
#   representan los patrones espaciales de la señal.
#
# En el contexto de la imagen térmica, el producto externo uᵢ·vᵢᵀ es el
# i-ésimo mapa de calor base y σᵢ es su peso. Los primeros tres modos capturan
# la zona caliente central, la anomalía secundaria y el gradiente diagonal con
# los que fue construida la señal limpia.
#
# -----------------------------------------------------------------------------
#
# Pregunta 2 — Cola plana del ruido en el Scree Plot
#
# El ruido blanco gaussiano es una matriz aleatoria con entradas i.i.d. de
# media cero. Según la teoría de matrices aleatorias (distribución de
# Marchenko-Pastur), los valores singulares de una matriz gaussiana m×n se
# distribuyen de forma aproximadamente continua y uniforme, formando una
# densidad en forma de semidisco — lo que se observa como una cola plana en
# el Scree Plot.
#
# La señal estructural, en cambio, tiene dependencia entre filas y columnas
# (rango bajo): solo unos pocos modos concentran casi toda la varianza,
# produciendo valores singulares dominantes que sobresalen por encima del piso
# del ruido. La transición entre la zona dominante y la cola plana es el codo,
# y constituye el criterio visual para elegir k.
#
# -----------------------------------------------------------------------------
#
# Pregunta 3 — Extremos: k = 1 y k = min(m, n)
#
# k = 1: A₁ = σ₁·u₁·v₁ᵀ es una matriz de rango 1, el producto externo de dos
# vectores. Solo captura el modo más energético (la gaussiana central). El
# resultado es una imagen muy borrosa que pierde la anomalía secundaria y el
# gradiente; retiene la estructura global pero descarta todo detalle.
#
# k = min(m,n) = 80: se usan todos los valores singulares disponibles, de modo
# que A_k = A_observada exactamente. No hay ningún filtrado: se incluye también
# todo el ruido, lo que equivale a no aplicar ningún procesamiento.
#
# La elección óptima de k está entre ambos extremos: suficientemente grande
# para capturar la señal estructural, suficientemente pequeño para excluir los
# modos dominados por ruido.
#
# -----------------------------------------------------------------------------
#
# Pregunta 4 — Umbral de mejora en la tabla de la Fase 4
#
# Observando la tabla, el error relativo disminuye rápidamente de k = 1 hasta
# k ≈ 5–10, y luego la mejora marginal por componente adicional se vuelve
# despreciable (del orden de 0.001 o menos). Este punto de saturación
# corresponde al umbral donde los valores singulares ya no representan señal
# estructural sino ruido.
#
# Físicamente, ese umbral marca la frontera entre los modos que describen el
# comportamiento termal real de la pieza y los modos generados por
# fluctuaciones aleatorias del sensor. Agregar más componentes incorpora ruido
# a la reconstrucción sin mejorar la fidelidad respecto a la señal limpia,
# equivalente a un overfitting sobre los datos ruidosos.
#
# -----------------------------------------------------------------------------
#
# Pregunta 5 — SVD reducida vs. SVD completa
#
# La SVD completa (full_matrices=True) calcula U ∈ ℝᵐˣᵐ y V ∈ ℝⁿˣⁿ completas,
# incluyendo vectores ortonormales adicionales que extienden las bases pero no
# contribuyen a la reconstrucción de A (corresponden al espacio nulo).
#
# La SVD reducida (full_matrices=False) calcula solo U ∈ ℝᵐˣʳ y Vᵀ ∈ ℝʳˣⁿ
# con r = min(m,n), eliminando las columnas y filas que multiplican los ceros
# de Σ. Para matrices con m >> n (muchas observaciones, pocas variables), el
# ahorro en memoria es significativo: U pasa de m×m a m×n, que en datos de
# sensores reales (por ejemplo, 10 000 × 50) representa una reducción de ×200.
# Para nuestro caso (80×100) la diferencia es pequeña, pero en aplicaciones
# industriales de monitoreo continuo esta distinción es crítica.
#
# =============================================================================